In [1]:
from sklearn.ensemble import RandomForestClassifier
import cv2 as cv
import numpy as np
import os
import matplotlib.pyplot as plt
from keras.layers import Input, Dense, Conv2D, MaxPool2D, GlobalAveragePooling2D
from keras.models import Sequential, Model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
data = []
data_dir = 'C:/Users/fahad/Desktop/Detection-of-Anemia-Using-Conjuctiva-Images-main/artifacts'
dirs = os.listdir(data_dir)
            
for i in dirs:
    path = os.path.join(data_dir,i)
    for img in os.listdir(path):
        image = cv.imread(os.path.join(path,img))
        image = cv.resize(image, (64,64))
        data.append([image, i])

In [3]:
os.listdir(data_dir)

['Resized Anemia', 'Resized Non Anemia']

In [4]:
images = []
labels = []

for img, lab in data:
    images.append(img)
    labels.append(lab)

In [5]:
from sklearn.preprocessing import LabelEncoder

In [6]:
images = np.array(images)

In [7]:
images = images/255.

In [8]:
le = LabelEncoder()
encoded = le.fit_transform(labels)

In [9]:
labels = np.array(labels)

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
x_train, x_test, y_train, y_test = train_test_split(images, encoded, test_size=0.1)

In [12]:
inputs = Input(shape=(64,64,3))

x1 = (Conv2D(32, (2,2), input_shape=(64,64,3), padding="same", activation="relu"))(inputs)
x2 = (MaxPool2D(2,2))(x1)
x3 = (Conv2D(64, (2,2), padding="same", activation="relu"))(x2)
x4 = (MaxPool2D(2,2))(x3)
x5 = (Conv2D(128, (2,2), padding="same", activation="relu"))(x4)
x6 = (MaxPool2D(2,2))(x5)

x7 = (GlobalAveragePooling2D())(x6)
x8 = (Dense(100, activation="relu"))(x7)
x9 = (Dense(2, activation="sigmoid"))(x8)

c:\Users\fahad\Desktop\Detection-of-Anemia-Using-Conjuctiva-Images-main\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model = Model(inputs=inputs, outputs=x9)

In [14]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        12,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           202 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,670 (213.55 KB)

 Trainable params: 54,670 (213.55 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=['accuracy'])

In [16]:
history = model.fit(x_train, y_train, validation_data=(x_test,y_test), epochs=40, batch_size=8)

Epoch 1/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.4864 - loss: 0.6998 - val_accuracy: 0.5909 - val_loss: 0.6754
Epoch 2/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5672 - loss: 0.6904 - val_accuracy: 0.5909 - val_loss: 0.6782
Epoch 3/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5200 - loss: 0.6955 - val_accuracy: 0.5909 - val_loss: 0.6743
Epoch 4/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4844 - loss: 0.7084 - val_accuracy: 0.5909 - val_loss: 0.6729
Epoch 5/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6133 - loss: 0.6689 - val_accuracy: 0.5909 - val_loss: 0.6707
Epoch 6/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5633 - loss: 0.6761 - val_accuracy: 0.5909 - val_loss: 0.6431
Epoch 7/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5759 - loss: 0.6506 - val_accuracy: 0.5909 - val_loss: 0.5541
Epoch 8/40
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6383 - loss: 0.6015 - val_accuracy: 0.8636 - v

In [17]:
temp = Model(inputs=inputs, outputs=x6)

In [18]:
temp.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 8, 8, 128)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 41,568 (162.38 KB)

 Trainable params: 41,568 (162.38 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
train_features = temp.predict(x_train)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


In [20]:
train_features.shape

(196, 8, 8, 128)

In [21]:
train_reshaped = train_features.reshape(train_features.shape[0], -1)

In [22]:
train_reshaped.shape

(196, 8192)

In [23]:
test_features = temp.predict(x_test)
test_reshaped = test_features.reshape(test_features.shape[0], -1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


In [24]:
test_reshaped.shape

(22, 8192)

In [25]:
rf = RandomForestClassifier()
rf.fit(train_reshaped, y_train)

RandomForestClassifier()

In [26]:
y_pred_rf = rf.predict(test_reshaped)

In [27]:
accuracy_score(y_pred_rf, y_test)

0.9545454545454546

In [28]:
confusion_matrix(y_pred_rf, y_test)

array([[ 8,  0],
       [ 1, 13]], dtype=int64)